# Harreman: Metabolic Exchange Inference

This tutorial demonstrates how to use Harreman for inferring metabolic exchange between spatially proximal cells.

Harreman analyzes correlations between metabolic gene expression and spatial proximity patterns to identify genes involved in cell-cell metabolic communication.

## Key Features:
- Metabolic gene expression analysis
- Spatial correlation-based exchange inference
- Cell type-specific metabolic interactions
- Statistical significance testing

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Import spatialvi
import spatialvi
from spatialvi.external import Harreman

sc.set_figure_params(figsize=(6, 6))
print("spatialvi version:", spatialvi.__version__)

## 1. Load Data

In [ ]:
# Load example spatial data
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()

print(adata)

In [ ]:
# Basic preprocessing
sc.pp.filter_genes(adata, min_cells=10)

# For this example, create synthetic cell type labels
np.random.seed(42)
cell_types = ["Epithelial", "Immune", "Stromal", "Endothelial"]
adata.obs["cell_type"] = pd.Categorical(np.random.choice(cell_types, adata.n_obs))

print("\nCell type distribution:")
print(adata.obs["cell_type"].value_counts())

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.spatial(adata, color="total_counts", spot_size=80, ax=axes[0], show=False)
sc.pl.spatial(adata, color="cell_type", spot_size=80, ax=axes[1], show=False)

plt.tight_layout()
plt.show()

## 2. Initialize Harreman Model

In [ ]:
# Initialize Harreman model
model = Harreman(
    adata,
    metabolic_genes=None,  # Use default metabolic gene set
    spatial_key="spatial",
    labels_key="cell_type",
    n_neighbors=20,  # Number of spatial neighbors
)

print(f"Number of metabolic genes found: {len(model.metabolic_genes)}")
print(f"\nFirst 10 metabolic genes: {model.metabolic_genes[:10]}")

## 3. Fit the Model

In [ ]:
# Fit the model
model.fit(
    n_permutations=100,  # Number of permutations for null distribution
    correlation_method="spearman",  # Correlation method
)

print("Model fitting complete!")

## 4. Get Metabolic Exchange Results

In [ ]:
# Get metabolic exchange results
exchanges = model.get_metabolic_exchanges(fdr_threshold=0.05)

print(f"Total metabolic genes analyzed: {len(exchanges)}")
print(f"Significant exchanges (FDR < 0.05): {(exchanges['fdr'] < 0.05).sum()}")

print("\nTop 15 spatially correlated metabolic genes:")
exchanges.head(15)

In [ ]:
# Visualize spatial correlation distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spatial correlation distribution
axes[0].hist(exchanges["spatial_correlation"], bins=30, edgecolor="black")
axes[0].axvline(0, color="red", linestyle="--", label="Zero")
axes[0].set_xlabel("Spatial Correlation")
axes[0].set_ylabel("Number of Genes")
axes[0].set_title("Distribution of Spatial Correlations")
axes[0].legend()

# Z-score distribution
axes[1].hist(exchanges["z_score"], bins=30, edgecolor="black")
axes[1].axvline(-1.96, color="red", linestyle="--", label="p=0.05")
axes[1].axvline(1.96, color="red", linestyle="--")
axes[1].set_xlabel("Z-Score")
axes[1].set_ylabel("Number of Genes")
axes[1].set_title("Distribution of Z-Scores")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Analyze Sender and Receiver Genes

In [ ]:
# Separate sender and receiver genes
significant = exchanges[exchanges["fdr"] < 0.1]

senders = significant[significant["exchange_type"] == "sender"]
receivers = significant[significant["exchange_type"] == "receiver"]

print(f"Sender genes (positive correlation): {len(senders)}")
print(f"Receiver genes (negative correlation): {len(receivers)}")

print("\nTop sender genes:")
if len(senders) > 0:
    print(senders[["gene", "spatial_correlation", "z_score", "fdr"]].head(10))

print("\nTop receiver genes:")
if len(receivers) > 0:
    print(receivers[["gene", "spatial_correlation", "z_score", "fdr"]].head(10))

In [ ]:
# Volcano plot
plt.figure(figsize=(10, 8))

# Plot all genes
plt.scatter(
    exchanges["spatial_correlation"],
    -np.log10(exchanges["fdr"] + 1e-10),
    c="gray",
    alpha=0.5,
    label="Not significant",
)

# Highlight significant genes
sig_mask = exchanges["fdr"] < 0.1
plt.scatter(
    exchanges.loc[sig_mask, "spatial_correlation"],
    -np.log10(exchanges.loc[sig_mask, "fdr"] + 1e-10),
    c="red",
    alpha=0.7,
    label="Significant (FDR < 0.1)",
)

# Add threshold line
plt.axhline(-np.log10(0.1), color="blue", linestyle="--", label="FDR = 0.1")

plt.xlabel("Spatial Correlation")
plt.ylabel("-log10(FDR)")
plt.title("Metabolic Exchange Volcano Plot")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Visualize Top Exchange Genes Spatially

In [ ]:
# Get top genes present in the data
top_genes = exchanges.head(6)["gene"].tolist()
available_genes = [g for g in top_genes if g in adata.var_names]

if len(available_genes) > 0:
    # Visualize spatially
    sc.pl.spatial(adata, color=available_genes, ncols=3, spot_size=60)
else:
    print("Top exchange genes not found in dataset")

## 7. Cell Type-Specific Exchanges

In [ ]:
# Get cell type-specific metabolic exchanges
ct_exchanges = model.get_cell_type_exchanges(min_cells=10)

print(f"Total cell type pair exchanges: {len(ct_exchanges)}")
ct_exchanges.head(20)

In [ ]:
# Aggregate by cell type pair
pair_summary = (
    ct_exchanges.groupby(["sender", "receiver"])
    .agg({"mean_expression": "mean", "n_cells": "mean", "gene": "count"})
    .reset_index()
)
pair_summary.columns = ["sender", "receiver", "mean_expr", "n_cells", "n_genes"]

print("Cell type pair summary:")
pair_summary

In [ ]:
# Create heatmap of cell type interactions
if len(ct_exchanges) > 0:
    # Pivot for heatmap
    heatmap_data = pair_summary.pivot(index="sender", columns="receiver", values="mean_expr").fillna(0)

    plt.figure(figsize=(10, 8))
    sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="YlOrRd")
    plt.title("Mean Metabolic Expression by Cell Type Pair")
    plt.xlabel("Receiver Cell Type")
    plt.ylabel("Sender Cell Type")
    plt.tight_layout()
    plt.show()

## 8. Store Results in AnnData

In [ ]:
# Store results in AnnData
model.to_adata()

# Check stored results
harreman_cols = [col for col in adata.var.columns if col.startswith("harreman_")]
print(f"Harreman results stored in adata.var: {harreman_cols}")

# Show results for metabolic genes
metabolic_results = adata.var.loc[adata.var["harreman_spatial_correlation"].notna(), harreman_cols].sort_values(
    "harreman_z_score", ascending=False
)

print(f"\nResults for {len(metabolic_results)} metabolic genes stored")
metabolic_results.head(10)

## 9. Pathway-Level Analysis

In [ ]:
# Group genes by metabolic pathway (simplified)
pathway_genes = {
    "Glycolysis": ["HK1", "HK2", "GPI", "PFKL", "ALDOA", "GAPDH", "PGK1", "ENO1", "PKM", "LDHA"],
    "TCA Cycle": ["CS", "ACO1", "IDH1", "IDH2", "OGDH", "SDHA", "FH", "MDH1", "MDH2"],
    "OXPHOS": ["NDUFA1", "NDUFB1", "UQCRC1", "COX4I1", "ATP5A1", "ATP5B"],
    "Amino Acid": ["GOT1", "GOT2", "GPT", "GLS", "GLUD1"],
    "Lipid": ["FASN", "ACACA", "SCD", "FADS1"],
    "Transporters": ["SLC2A1", "SLC2A3", "SLC16A1", "SLC1A5"],
}

# Compute pathway-level statistics
pathway_stats = []
for pathway, genes in pathway_genes.items():
    pathway_data = exchanges[exchanges["gene"].isin(genes)]
    if len(pathway_data) > 0:
        pathway_stats.append(
            {
                "pathway": pathway,
                "n_genes": len(pathway_data),
                "mean_correlation": pathway_data["spatial_correlation"].mean(),
                "mean_zscore": pathway_data["z_score"].mean(),
                "n_significant": (pathway_data["fdr"] < 0.1).sum(),
            }
        )

pathway_df = pd.DataFrame(pathway_stats)
print("Pathway-level summary:")
pathway_df.sort_values("mean_zscore", ascending=False)

In [ ]:
# Visualize pathway-level results
if len(pathway_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Mean Z-score by pathway
    pathway_df.sort_values("mean_zscore").plot(kind="barh", x="pathway", y="mean_zscore", ax=axes[0], legend=False)
    axes[0].set_xlabel("Mean Z-Score")
    axes[0].set_title("Spatial Correlation by Pathway")
    axes[0].axvline(0, color="red", linestyle="--")

    # Number of significant genes
    pathway_df.sort_values("n_significant").plot(kind="barh", x="pathway", y="n_significant", ax=axes[1], legend=False)
    axes[1].set_xlabel("Number of Significant Genes")
    axes[1].set_title("Significant Genes by Pathway")

    plt.tight_layout()
    plt.show()

## Summary

In this tutorial, we demonstrated:

1. How to prepare spatial data for Harreman analysis
2. How to initialize and fit the Harreman model
3. How to identify spatially correlated metabolic genes
4. How to distinguish sender vs receiver genes
5. How to analyze cell type-specific metabolic exchanges
6. How to perform pathway-level analysis

Harreman provides insights into metabolic communication between spatially proximal cells, helping understand tissue metabolism and cell-cell metabolic dependencies.